# Dump of the FEM/FDM code, which we did not use in the end

In [ ]:
import numpy as np
import torch
from scipy.sparse import diags, kron, eye, csr_matrix
from scipy.sparse.linalg import spsolve

def solve_FD(theta):
    is_tensor = isinstance(theta, torch.Tensor)
    a = theta.numpy().astype(float) if is_tensor else np.asarray(theta, dtype=float)

    N = a.shape[0]
    h = 1.0 / (N - 1)
    n = N - 2  # number of interior points per dimension

    # ── Face-centred permeabilities (harmonic mean at interfaces) ─────────────
    # x-direction interfaces: between (i, j) and (i, j+1), for interior columns
    # shape: (n, n-1) — there are n-1 x-interfaces per interior row
    ax = a[1:-1, :]  # all columns including boundary, interior rows
    a_x = 2 * ax[:, :-1] * ax[:, 1:] / (ax[:, :-1] + ax[:, 1:] + 1e-30)
    # x-interfaces touching interior points only: columns 0..n-1 vs 1..n
    # i.e. interface between col j and j+1 for j=0..N-2
    # We need interfaces (j=0,interior) to (j=N-2, interior)
    # For interior x: between interior col j and j+1: j=0..n-2 → shape (n, n-1)
    a_x_int = a_x[:, 1:-1]  # interfaces strictly between interior points, shape (n, n-1)... 
    
    # Actually let's be more explicit:
    # a_e[i,j] = harmonic mean of a[i,j] and a[i,j+1] for interior i,j
    # using full grid indices (rows 1..N-2, cols 1..N-2)
    a_full = a  # shape (N, N)
    
    # East interface: between interior point (i,j) and (i,j+1)
    # For j < n-1 (not the last interior column), neighbour is interior
    # For j = n-1, neighbour is boundary (a=0 enforced by BC, not permeability)
    # Permeability at east face of interior point (i,j): harmonic mean of a[i+1,j+1] and a[i+1,j+2]
    # (shifting to full grid: interior point (i,j) is at full grid (i+1, j+1))
    
    def harmonic(x, y):
        return 2 * x * y / (x + y + 1e-30)

    # Permeability faces for interior grid points (i,j), i,j = 0..n-1
    # Full grid index: (i+1, j+1)
    i_idx = np.arange(1, N-1)  # interior rows in full grid
    j_idx = np.arange(1, N-1)  # interior cols in full grid

    # East face: between (i+1,j+1) and (i+1,j+2)
    ae = harmonic(a_full[1:-1, 1:-1], a_full[1:-1, 2:  ])  # shape (n,n), but last col has boundary neighbour
    # West face: between (i+1,j+1) and (i+1,j)
    aw = harmonic(a_full[1:-1, 1:-1], a_full[1:-1,  :-2])  # shape (n,n)
    # North face: between (i+1,j+1) and (i+2,j+1)
    an = harmonic(a_full[1:-1, 1:-1], a_full[2:,   1:-1])  # shape (n,n)
    # South face: between (i+1,j+1) and (i,j+1)
    as_ = harmonic(a_full[1:-1, 1:-1], a_full[ :-2, 1:-1])  # shape (n,n)

    # ── Build 1D tridiagonal operators then Kronecker product ─────────────────
    # x-direction operator (n x n tridiagonal per row)
    # For each row i, the x-contribution is:
    # (1/h²) * [-aw[i,j]*u[i,j-1] + (aw[i,j]+ae[i,j])*u[i,j] - ae[i,j]*u[i,j+1]]
    # This is NOT a simple Kronecker product because ae/aw vary per row.
    # So build the full n²×n² matrix directly via diags on the flattened system.

    n2 = n * n
    
    # Flatten row-major: index k = i*n + j
    ae_flat  = ae.ravel()   # east permeability for each interior point
    aw_flat  = aw.ravel()   # west
    an_flat  = an.ravel()   # north
    as_flat  = as_.ravel()  # south

    # Centre coefficient
    diag_main = (ae_flat + aw_flat + an_flat + as_flat) / h**2

    # East neighbour: k → k+1, but zero if j=n-1 (right boundary)
    east = ae_flat / h**2
    east_mask = np.ones(n2)
    east_mask[n-1::n] = 0  # last column of each row has no east interior neighbour
    east = east * east_mask

    # West neighbour: k → k-1, but zero if j=0 (left boundary)  
    west = aw_flat / h**2
    west_mask = np.ones(n2)
    west_mask[::n] = 0  # first column of each row has no west interior neighbour
    west = west * west_mask

    # North neighbour: k → k+n
    north = an_flat / h**2

    # South neighbour: k → k-n
    south = as_flat / h**2

    # Assemble
    A = (
        diags(-diag_main,        0,  shape=(n2, n2))
      + diags(east[:-1],         1,  shape=(n2, n2))
      + diags(west[1:],         -1,  shape=(n2, n2))
      + diags(north[:-n],        n,  shape=(n2, n2))
      + diags(south[n:],        -n,  shape=(n2, n2))
    ).tocsr()

    # ── RHS: f = 1 ────────────────────────────────────────────────────────────
    rhs = np.ones(n2)

    # ── Solve ─────────────────────────────────────────────────────────────────
    u_int = spsolve(A, rhs)
    u_int = u_int.reshape(n, n)

    # ── Pad with boundary zeros ───────────────────────────────────────────────
    u = np.zeros((N, N))
    u[1:-1, 1:-1] = u_int

    if is_tensor:
        return torch.tensor(u, dtype=theta.dtype)
    return u.astype(np.float64)

In [ ]:
import numpy as np
from mpi4py import MPI
from dolfinx import mesh, fem
from dolfinx.fem.petsc import LinearProblem
import ufl
from scipy.interpolate import griddata
import torch

def solve_FEM(theta):
    """
    Solve Darcy flow PDE given a discretised permeability field.
    
    Parameters
    ----------
    theta : np.ndarray or torch.Tensor of shape (H, W)
        Binary/continuous permeability field (values in {3, 12} or similar).
        The mesh resolution is inferred adaptively from the input shape.
    
    Returns
    -------
    u_out : same type and shape as theta
        The pressure/flow field solving -∇·(a(x)∇u) = 1 on [0,1]²
        with homogeneous Dirichlet BCs.
    """
    is_tensor = isinstance(theta, torch.Tensor)
    theta_np = theta.numpy() if is_tensor else np.asarray(theta)

    H, W = theta_np.shape  # infer resolution from input

    # ── 1. MESH ───────────────────────────────────────────────────────────────
    domain = mesh.create_unit_square(MPI.COMM_WORLD, H-1, W-1)

    # ── 2. PERMEABILITY FIELD a(x) ────────────────────────────────────────────
    # DG0: piecewise constant per triangle cell
    V_k = fem.functionspace(domain, ("DG", 0))
    a_func = fem.Function(V_k)

    tdim = domain.topology.dim
    num_cells = domain.topology.index_map(tdim).size_local
    midpoints = mesh.compute_midpoints(domain, tdim, np.arange(num_cells))

    # Map each cell midpoint → pixel index in theta
    ix = np.clip((midpoints[:, 0] * W).astype(int), 0, W)
    iy = np.clip((midpoints[:, 1] * H).astype(int), 0, H)

    # Permeability: high (1.0) where theta is large, low (1e-3) elsewhere
    threshold = 0.5 * (theta_np.min() + theta_np.max())
    a_func.x.array[:] = np.where(theta_np[iy, ix] > threshold, 1.0, 1e-3)

    # ── 3. FUNCTION SPACE & BCs ───────────────────────────────────────────────
    V = fem.functionspace(domain, ("Lagrange", 1))

    fdim = tdim - 1
    all_boundary_facets = mesh.locate_entities_boundary(
        domain, fdim, lambda x: np.full(x.shape[1], True, dtype=bool)
    )
    bc = fem.dirichletbc(
        fem.Constant(domain, 0.0),
        fem.locate_dofs_topological(V, fdim, all_boundary_facets),
        V,
    )

    # ── 4. VARIATIONAL FORM ───────────────────────────────────────────────────
    # -∇·(a(x)∇u) = 1  →  ∫ a ∇u·∇v dx = ∫ v dx
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    a_form = a_func * ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L_form = v * ufl.dx

    # ── 5. SOLVE ──────────────────────────────────────────────────────────────
    problem = LinearProblem(
        a_form, L_form, bcs=[bc],
        petsc_options_prefix="darcy",
        petsc_options={
            "ksp_type": "cg",
            "pc_type": "hypre",
            "pc_hypre_type": "boomeramg",
            "ksp_rtol": 1e-12,
        },
    )
    uh = problem.solve()

    # ── 6. INTERPOLATE BACK ONTO H×W GRID ────────────────────────────────────
    coords = V.tabulate_dof_coordinates()[:, :2]
    u_vals = uh.x.array.real

    grid_x, grid_y = np.meshgrid(
        np.linspace(0, 1, W), np.linspace(0, 1, H)
    )
    u_grid = griddata(coords, u_vals, (grid_x, grid_y), method="linear")

    # Fill any boundary NaNs from griddata with 0 (matches Dirichlet BC)
    u_grid = np.nan_to_num(u_grid, nan=0.0)

    # ── 7. RETURN SAME TYPE AS INPUT ──────────────────────────────────────────
    if is_tensor:
        return torch.tensor(u_grid, dtype=theta.dtype)
    return u_grid.astype(theta_np.dtype)
